# Virasoro fusion at $c = 0$: Eberle–Flohr and Gaberdiel–Runkel–Wood

Eberle and Flohr [EF2006] used the Nahm–Gaberdiel–Kausch algorithm [Nah1994, GK1996] to compute many fusion products of the universal Virasoro vertex algebra at $c = c_{2,3} = 0$. These are products of the irreducible modules $\mathcal V(h)$, for $h$ in the extended Kac table, and they give indecomposable modules of rank 2 and rank 3, the rank being the size of the largest Jordan block of $L_0$. Eberle and Flohr also computed the logarithmic couplings $\beta$ of the modules of rank 2 [EF2006, Table 2]. Gaberdiel, Runkel and Wood [GRW2009, §2.2] corrected one of their results: $\mathcal V(2) \boxtimes \mathcal V(2)$ is not the vacuum module $\mathcal V$ but its contragredient module $\mathcal V^*$, and the products of $\mathcal V(1)$ change likewise. Here $M(h)$ is the Verma module of lowest weight $h$, $\mathcal V(h)$ is its irreducible quotient, and $\mathcal V = M(0)/\langle L_{-1}\Omega\rangle$ is the vacuum module of the universal Virasoro vertex algebra.

This notebook reproduces these computations with `M.fusion_product(N)` (`11_fusion_and_extensions.ipynb`, sections 1 and 2), which computes the depth-$d$ quotients $(M \boxtimes N)^{(d)}$: the quotient of $M \boxtimes N$ by the words of modes of total degree $> d$, where the mode $L_{-n}$ has degree $n$. These quotients carry the action of $\Delta(L_0)$, whose Jordan blocks show the rank, the action of the modes of degree $0, \dots, d$, and the lowering modes $L_n$ as maps to the quotients of depth $d - n$. The indecomposable modules are written as in [EF2006]: $\mathcal R^{(2)}(h_1, h_2)$ for rank 2 and $\mathcal R^{(3)}(h_1, h_1, h_2, h_2)$ for rank 3.

**Prerequisites:** `10_modules.ipynb`, `11_fusion_and_extensions.ipynb`.

**Elsewhere:** the $(2, 3)$ triplet algebra, of central charge $c = 0$, is constructed in `14_triplet_algebra.ipynb`.

**Contents**

1. The window
2. The correction of Gaberdiel–Runkel–Wood
3. Eberle–Flohr's table at depth 0
4. Rank 2: the logarithmic couplings
5. Rank 3
6. Limitations

The cell below defines the universal Virasoro vertex algebra at $c = 0$, a function `irr(h)` that returns the irreducible module $\mathcal V(h)$ (the simple quotient of the Verma module, cached), and the entries $h_{r,s} = ((3r - 2s)^2 - 1)/24$ of the Kac table of $c_{2,3} = 0$. It prints the entries $h_{1,s}$ and $h_{2,s}$ for $s = 1, \dots, 7$.

In [1]:
from vertex_algebras import *
import time
import vertex_algebras as va
V = va.Virasoro(QQ, 0, conformal_vector='L'); L = V.gen('L')
_irr = {}


def irr(h):
    """The irreducible module V(h) (the simple quotient of the Verma module)."""
    if h not in _irr:
        _irr[h] = V.verma_module(h).simple_quotient()
    return _irr[h]


def kac(r, s):
    return ((3*r - 2*s)^2 - 1) / 24


print([kac(1, s) for s in range(1, 8)], [kac(2, s) for s in range(1, 8)])

[0, 0, 1/3, 1, 2, 10/3, 5] [5/8, 1/8, -1/24, 1/8, 5/8, 35/24, 21/8]


## 1. The window

The quotients are computed from the tensors $x \otimes y$ of total depth $\le K$, where the depth of a vector is its weight minus the lowest conformal weight of its module (`11_fusion_and_extensions.ipynb`, section 2). Once $K$ is large enough, the result maps onto the true quotient, and it only shrinks as $K$ grows. Two things must fit in the window:

* Nahm's special subspace of the first factor $\mathcal V(h_1)$ up to depth $d$, which ends below the first null vector of $\mathcal V(h_1)$;
* the null vectors that generate the maximal submodules of the Verma modules of both factors, since their relations are what cut the product down. Eberle and Flohr make the same remark about their $\tilde L_{\max}$ [EF2006].

Stability over a few consecutive windows is *not* enough: below these levels, a spurious state can survive several windows.

The generating null vectors are read off the characters. The first one is at the first level $l_1$ with $\dim \mathcal V(h)_{h+n} < p(n)$, where $p(n)$ is the number of partitions of $n$. For the modules of braid type, in the bulk of the Kac table, the maximal submodule needs a second generator, at the first level $l_2 > l_1$ with $\dim \mathcal V(h)_{h+n} < p(n) - p(n - l_1)$. The cell below defines `generating_null_levels(h)`, which finds these levels up to level 14, and `window(h1, h2, d)`, which returns a window containing both the special subspace and the generating null vectors. It prints the levels for the weights used below.

In [2]:
def generating_null_levels(h, bound=14):
    """The levels of the null vectors generating the maximal submodule of M(h), up to bound."""
    Lh = irr(h)
    dims = [Lh.weight_space(h + n).dimension() for n in range(bound + 1)]
    p = [Partitions(n).cardinality() for n in range(bound + 1)]
    l1 = next((n for n in range(bound + 1) if dims[n] < p[n]), None)
    if l1 is None:
        return []
    l2 = next((n for n in range(l1 + 1, bound + 1) if dims[n] < p[n] - p[n - l1]), None)
    return [l1] if l2 is None else [l1, l2]


def window(h1, h2, d):
    """A window containing the special subspace of V(h1) at depth d and the generating null vectors."""
    g1, g2 = generating_null_levels(h1), generating_null_levels(h2)
    return max([d + 1 + (g1[0] if g1 else 0)] + g1 + g2) + 1


print({h: generating_null_levels(h) for h in (0, 1/3, 5/8, 1/8, -1/24, 2, 1, 5, 7, 10/3, 21/8, 33/8, 35/24)})

{0: [1, 2], 1/3: [3], 5/8: [2], 1/8: [4], -1/24: [6], 2: [3, 5], 1: [4, 6], 5: [7, 10], 7: [5, 8], 10/3: [6], 21/8: [8], 33/8: [4], 35/24: [12]}


## 2. The correction of Gaberdiel–Runkel–Wood

At $c = 0$ the irreducible vacuum module $\mathcal V(0)$ is one-dimensional, since both $L_{-1} \Omega$ and $L_{-2} \Omega$ are null. Hence $\mathcal V(0) \boxtimes \mathcal V(0) = \mathcal V(0)$, and $\mathcal V(0) \boxtimes \mathcal V(h) = 0$ for $h \ne 0$ [GRW2009, (2.8)]. Eberle and Flohr found $\mathcal V(2) \boxtimes \mathcal V(2) = \mathcal V$ (their $\mathcal R^{(1)}(0)_2$). With associativity, this would give the contradiction $\mathcal V(0) = \mathcal V(0) \boxtimes \mathcal V = (\mathcal V(0) \boxtimes \mathcal V(2)) \boxtimes \mathcal V(2) = 0$.

The cell below first prints the dimensions of the weight spaces of $\mathcal V(0)$ in the weights $0, \dots, 3$, the conformal weights of the depth-0 quotient of $\mathcal V(0) \boxtimes \mathcal V(0)$, and the dimension of the depth-0 quotient of $\mathcal V(0) \boxtimes \mathcal V(2)$. The depth-0 quotient of $\mathcal V$ is spanned by $\Omega$ alone, but the cell then shows that the depth-0 quotient of $\mathcal V(2) \boxtimes \mathcal V(2)$ is two-dimensional, with $\Delta(L_0)$ conjugate to $\mathrm{diag}(0, 2)$ [GRW2009, (2.16), (2.17)].

In [3]:
print([irr(0).weight_space(n).dimension() for n in range(4)])
print(irr(0).fusion_product(irr(0)).depth_quotient_up_to(0, 3).conformal_weights(),
      irr(0).fusion_product(irr(2)).depth_quotient_up_to(0, 6).dimension())
F = irr(2).fusion_product(irr(2)).depth_quotient_up_to(0, window(2, 2, 0))
print(F.basis())
print(F.l0_matrix(), F.conformal_weights())

[1, 0, 0, 0]
[(0, [1])] 0
[(v, v), (L_{-1} v, v)]
[ 4 -8]
[ 1 -2] [(0, [1]), (2, [1])]


Gaberdiel, Runkel and Wood also checked that $L_2$ maps the state of weight 2 to the state of weight 0, as in $\mathcal V^*$; in $\mathcal V$ itself, $L_2 L_{-2} \Omega = \frac c2 \Omega = 0$. In the depth quotients, $L_2$ maps the depth-2 quotient to the depth-0 quotient. The cell below finds a state of weight 2 in the depth-2 quotient that is not the image of a raising mode (its projection to the depth-0 quotient is nonzero), and checks that $L_2$ maps it to a nonzero multiple of the state of weight 0.

In [4]:
P22 = irr(2).fusion_product(irr(2))
K = window(2, 2, 2)
F2, F0 = P22.depth_quotient_up_to(2, K), P22.depth_quotient_up_to(0, K)
L0 = F2.l0_matrix()
print(F2.conformal_weights())
w2 = [v for v in (L0 - 2).right_kernel().basis() if (F2.projection(0) * v) != 0]   # weight 2, not a raising image
w0 = (L0 - 0).right_kernel().basis()[0]
a, b = F2.mode_matrix(L, 3) * w2[0], F2.projection(0) * w0
print(a != 0, matrix([a, b]).rank() == 1)        # L_2 (weight 2) = nonzero multiple of the weight-0 state

[(0, [1]), (2, [1]), (3, [1]), (4, [1, 1])]
True True


**The products of $\mathcal V(1)$.** Gaberdiel, Runkel and Wood find $\mathcal V(1) \boxtimes \mathcal V(2) = \mathcal P^*$ and $\mathcal V(1) \boxtimes \mathcal V(1) = \mathcal V^* \oplus \mathcal V(\frac13)$ [GRW2009], where $\mathcal P^*$ is the contragredient of their module $\mathcal P$, instead of Eberle and Flohr's $\mathcal R^{(1)}(0)_1$ and $\mathcal R^{(1)}(0)_2 \oplus \mathcal V(\frac13)$. The cell below shows that the depth-0 quotients have the weights $0, 1$ and $0, \frac13, 2$.

In [5]:
for h1, h2 in [(1, 2), (1, 1)]:
    print(h1, h2, irr(h1).fusion_product(irr(h2)).depth_quotient_up_to(0, window(h1, h2, 0)).conformal_weights())

1 2 [(0, [1]), (1, [1])]


1 1 [(0, [1]), (1/3, [1]), (2, [1])]


## 3. Eberle–Flohr's table at depth 0

This section checks the rows of $\mathcal V(\frac58)$, $\mathcal V(\frac13)$, $\mathcal V(\frac18)$ and $\mathcal V(2)$ in [EF2006, Appendix B]. At depth 0, each summand of a fusion product shows its generating states:

* $\mathcal V(h)$: one state of weight $h$;
* a rank-2 module $\mathcal R^{(2)}(h, h)$ (type A): a Jordan block of size 2 at $h$;
* a rank-2 module $\mathcal R^{(2)}(h_1, h_2)$ with $h_1 < h_2$: two states, of weights $h_1$ and $h_2$, the ground state and the logarithmic partner. The Jordan block pairs the partner with a descendant of the ground state, which vanishes in the depth-0 quotient, so no Jordan block is visible at depth 0 (section 4 finds it at depth $h_2 - h_1$);
* a rank-3 module $\mathcal R^{(3)}(h_1, h_1, h_2, h_2)$: Jordan blocks of size 2 at $h_1$ and at $h_2$.

The cell below encodes Eberle and Flohr's results as lists of summands, and converts each to the expected depth-0 data with `depth0`. It computes each product in the window of section 1, compares the result with that in the next window, of size $K + 1$, and compares both with Eberle and Flohr. All 38 products agree; the whole table takes about ten minutes.

In [6]:
V_ = lambda h: ('V', (h,)); R2 = lambda a, b: ('R2', (a, b)); R3 = lambda *w: ('R3', w)


def depth0(res):
    """Eberle-Flohr's result -> the depth-0 data [(h, Jordan block sizes)]."""
    out = {}
    for kind, ws in res:
        if kind == 'V':
            out.setdefault(ws[0], []).append(1)
        elif kind == 'R2' and ws[0] == ws[1]:
            out.setdefault(ws[0], []).append(2)
        elif kind == 'R2':
            for w in ws:
                out.setdefault(w, []).append(1)
        else:
            for w in sorted(set(ws)):
                out.setdefault(w, []).append(2)
    return sorted((w, sorted(s, reverse=True)) for w, s in out.items())


table = [
    (5/8, 5/8, [R2(0, 2)]), (5/8, 1/3, [V_(-1/24)]), (5/8, 1/8, [R2(0, 1)]), (5/8, -1/24, [R2(1/3, 1/3)]),
    (5/8, 33/8, [R2(2, 7)]), (5/8, 10/3, [V_(35/24)]), (5/8, 21/8, [R2(1, 5)]), (5/8, 35/24, [R2(1/3, 10/3)]),
    (5/8, 2, [V_(5/8)]), (5/8, 1, [V_(1/8)]), (5/8, 7, [V_(33/8)]), (5/8, 5, [V_(21/8)]),
    (1/3, 1/3, [R2(0, 2), V_(1/3)]), (1/3, 1/8, [R2(1/8, 1/8)]), (1/3, -1/24, [R2(5/8, 5/8), V_(-1/24)]),
    (1/3, 33/8, [V_(35/24)]), (1/3, 10/3, [R2(1, 7), V_(10/3)]), (1/3, 21/8, [R2(5/8, 21/8)]),
    (1/3, 35/24, [R2(1/8, 33/8), V_(35/24)]), (1/3, 2, [V_(1/3)]), (1/3, 1, [R2(0, 1)]), (1/3, 7, [V_(10/3)]),
    (1/3, 5, [R2(2, 5)]),
    (1/8, 1/8, [R2(1/3, 1/3), R2(0, 2)]), (1/8, -1/24, [R3(0, 0, 1, 1)]), (1/8, 33/8, [R2(1, 5)]),
    (1/8, 10/3, [R2(5/8, 21/8)]), (1/8, 21/8, [R2(1/3, 10/3), R2(2, 7)]), (1/8, 2, [V_(1/8)]),
    (1/8, 1, [V_(5/8), V_(-1/24)]), (1/8, 7, [V_(21/8)]), (1/8, 5, [V_(33/8), V_(35/24)]),
    (2, -1/24, [V_(-1/24)]), (2, 33/8, [V_(33/8)]), (2, 10/3, [V_(10/3)]), (2, 21/8, [V_(21/8)]),
    (2, 7, [V_(7)]), (2, 5, [V_(5)]),
]
t0 = time.time()
agree = 0
for h1, h2, res in table:
    P = irr(h1).fusion_product(irr(h2))
    K = window(h1, h2, 0)
    got = P.depth_quotient_up_to(0, K).conformal_weights()
    stable = got == P.depth_quotient_up_to(0, K + 1).conformal_weights()
    ok = got == depth0(res) and stable
    agree += ok
    print('V(%s) x V(%s)  K = %s:  %s%s' % (h1, h2, K, got, '' if ok else '   <- Eberle-Flohr: %s' % depth0(res)))
print('%s of %s agree (%.0f s)' % (agree, len(table), time.time() - t0))

V(5/8) x V(5/8)  K = 4:  [(0, [1]), (2, [1])]
V(5/8) x V(1/3)  K = 4:  [(-1/24, [1])]


V(5/8) x V(1/8)  K = 5:  [(0, [1]), (1, [1])]


V(5/8) x V(-1/24)  K = 7:  [(1/3, [2])]
V(5/8) x V(33/8)  K = 5:  [(2, [1]), (7, [1])]


V(5/8) x V(10/3)  K = 7:  [(35/24, [1])]


V(5/8) x V(21/8)  K = 9:  [(1, [1]), (5, [1])]


V(5/8) x V(35/24)  K = 13:  [(1/3, [1]), (10/3, [1])]
V(5/8) x V(2)  K = 6:  [(5/8, [1])]


... (17 outputs left out)

V(2) x V(10/3)  K = 7:  [(10/3, [1])]


V(2) x V(21/8)  K = 9:  [(21/8, [1])]


V(2) x V(7)  K = 9:  [(7, [1])]


V(2) x V(5)  K = 11:  [(5, [1])]
38 of 38 agree (44 s)


## 4. Rank 2: the logarithmic couplings

A rank-2 module $\mathcal R^{(2)}(h, h + l)$ of one of the types B–F of [EF2006] has a ground state $|h\rangle$, a singular vector $\chi = X |h\rangle$ at level $l$, normalised so that the word $X$ has the coefficient 1 at $L_{-1}^l$, and a logarithmic partner $\xi$ with $L_0 \xi = (h + l) \xi + \chi$. Eberle and Flohr define the logarithmic coupling $\beta$ by $L_1^l \xi = \beta |h\rangle$, after adding descendants of $|h\rangle$ to $\xi$ so that $L_p \xi = 0$ for $p \ge 2$. The same number is given without this choice of gauge by $X^\dagger \xi = \beta |h\rangle$, where $L_{-n}^\dagger = L_n$ [KyR2009]: $X^\dagger$ kills the descendants of $|h\rangle$ at level $l$, since $\chi$ is singular. For $\mathcal R^{(2)}(0, 2)$, where $L_{-1}|0\rangle$ is null, Eberle and Flohr use $X = L_{-2}$. The whole computation takes place in the depth-$l$ quotient: $X$ is a word of modes of degree $\le l$, and $X^\dagger$ maps the depth-$l$ quotient to the depth-0 quotient.

The cell below defines `singular_word(h, l)`, the word $X$ of the singular vector of the Verma module $M(h)$ at level $l$, as a dict over the sequences $(n_1, \dots, n_k)$ that stand for $L_{-n_1} \cdots L_{-n_k}$. It also defines `beta(h1, h2, h, l, K, X)`, which computes $\beta$ for the summand $\mathcal R^{(2)}(h, h+l)$ of $\mathcal V(h_1) \boxtimes \mathcal V(h_2)$ in the window $K$. It prints the words $L_{-1}^2 - \frac32 L_{-2}$ for $h = \frac58$, $l = 2$, and $L_{-1}$ for $h = 0$, $l = 1$.

In [7]:
def singular_word(h, l):
    """X with X v the singular vector of the Verma module M(h) at level l, normalised at L_{-1}^l: {(n_1, ..., n_k): c} for L_{-n_1} ... L_{-n_k}."""
    Mh = V.verma_module(h)
    B = list(Mh.weight_space(h + l).basis())
    imgs = [{(m, k): c for m in (2, 3) for k, c in L.nproduct(b, m).monomial_coefficients().items()} for b in B]   # L_1, L_2
    cols = sorted({k for r in imgs for k in r}, key=repr)
    x, = matrix(QQ, [[r.get(k, 0) for k in cols] for r in imgs]).left_kernel().basis()
    words = {tuple(-m for (i, m) in b.leading_support()[0]): c for b, c in zip(B, x) if c}
    return {w: c / words[(1,) * l] for w, c in words.items()}


def beta(h1, h2, h, l, K, X=None):
    """The logarithmic coupling of the rank-2 summand R(h, h + l) of V(h1) x V(h2)."""
    P = irr(h1).fusion_product(irr(h2))
    F = P.depth_quotient_up_to(l, K)
    L0 = F.l0_matrix()
    Omega, = (L0 - h).right_kernel().basis()
    X = singular_word(h, l) if X is None else X
    chi = 0
    for w, c in X.items():
        v = Omega
        for n in reversed(w):
            v = F.mode_matrix(L, 1 - n) * v              # L_{-n} = L_(1-n)
        chi += c * v
    xi = (L0 - (h + l)).solve_right(chi)
    out = 0
    for w, c in X.items():                               # (L_{-n_1} ... L_{-n_k})^† = L_{n_k} ... L_{n_1}
        v, G = xi, F
        for n in w:
            v, G = G.mode_matrix(L, n + 1) * v, P.depth_quotient_up_to(G.depth() - n, K)
        out += c * v
    o = F.projection(0) * Omega
    b = out[o.nonzero_positions()[0]] / o[o.nonzero_positions()[0]]
    assert out == b * o
    return b


print(singular_word(5/8, 2), singular_word(0, 1))

{(2,): -3/2, (1, 1): 1} {(1,): 1}


The cell below computes $\beta$ for seven rank-2 summands, with $X = L_{-2}$ for $\mathcal R^{(2)}(0, 2)$, and compares each value with [EF2006, Table 2]. All seven agree.

In [8]:
cases = [  # (h1, h2, h, l, window, Eberle-Flohr's beta, X)
    (5/8, 5/8, 0, 2, 6, 5/6, {(2,): 1}),        # R(0, 2)_7
    (1/3, 1/3, 0, 2, 6, -5/8, {(2,): 1}),       # R(0, 2)_5
    (5/8, 1/8, 0, 1, 5, 1/3, None),             # R(0, 1)_5
    (1/3, 1, 0, 1, 7, -1/2, None),              # R(0, 1)_7
    (1/3, 21/8, 5/8, 2, 9, -5, None),           # R(5/8, 21/8)
    (5/8, 35/24, 1/3, 3, 9, 140/27, None),      # R(1/3, 10/3)
    (1/3, 5, 2, 3, 9, -420, None),              # R(2, 5)
]
for h1, h2, h, l, K, expected, X in cases:
    t = time.time()
    b = beta(h1, h2, h, l, K, X)
    print('V(%s) x V(%s): R(%s, %s)  beta = %s  (Eberle-Flohr: %s)  %.0f s' % (h1, h2, h, h + l, b, expected, time.time() - t))

V(5/8) x V(5/8): R(0, 2)  beta = 5/6  (Eberle-Flohr: 5/6)  0 s
V(1/3) x V(1/3): R(0, 2)  beta = -5/8  (Eberle-Flohr: -5/8)  0 s


V(5/8) x V(1/8): R(0, 1)  beta = 1/3  (Eberle-Flohr: 1/3)  0 s


V(1/3) x V(1): R(0, 1)  beta = -1/2  (Eberle-Flohr: -1/2)  0 s


V(1/3) x V(21/8): R(5/8, 21/8)  beta = -5  (Eberle-Flohr: -5)  4 s


V(5/8) x V(35/24): R(1/3, 10/3)  beta = 140/27  (Eberle-Flohr: 140/27)  6 s


V(1/3) x V(5): R(2, 5)  beta = -420  (Eberle-Flohr: -420)  9 s


The two couplings at level 4 need the window $K = 12$ and take 10–15 minutes each; set `SLOW = True` in the cell below to recompute them. Computed on 25 October 2026: $\beta = \frac{2800}9$ for $\mathcal R^{(2)}(1, 5)$ in $\mathcal V(\frac58) \boxtimes \mathcal V(\frac{21}8)$ (603 s), and $\beta = -\frac{700}{81}$ for $\mathcal R^{(2)}(\frac18, \frac{33}8)$ in $\mathcal V(\frac13) \boxtimes \mathcal V(\frac{35}{24})$ (911 s), both as in [EF2006, Table 2]. The two-parameter modules $\mathcal R^{(2)}(1, 7)$ and $\mathcal R^{(2)}(2, 7)$ (types E and F) are not attempted, since their second coupling $\beta_2$ needs Eberle and Flohr's gauge.

In [9]:
SLOW = False
if SLOW:
    for h1, h2, h, l, K, expected in [(5/8, 21/8, 1, 4, 12, 2800/9), (1/3, 35/24, 1/8, 4, 12, -700/81)]:
        print(h1, h2, beta(h1, h2, h, l, K), expected)

## 5. Rank 3

Modules of rank 3 appear in two of the products: $\mathcal V(\frac18) \boxtimes \mathcal V(-\frac1{24}) = \mathcal R^{(3)}(0, 0, 1, 1)$ and $\mathcal V(-\frac1{24}) \boxtimes \mathcal V(-\frac1{24}) = \mathcal R^{(3)}(0, 0, 2, 2) \oplus \mathcal R^{(2)}(\frac13, \frac13)$ [EF2006]. The cell below computes the depth-1 quotient of the first product and the depth-2 quotient of the second. They have Jordan blocks of size 3 at weight 1 (depth 1) and at weight 2 (depth 2), respectively.

In [10]:
for h1, h2, d in [(1/8, -1/24, 1), (-1/24, -1/24, 2)]:
    t = time.time()
    F = irr(h1).fusion_product(irr(h2)).depth_quotient_up_to(d, window(h1, h2, d))
    print('V(%s) x V(%s), depth %s: %s  (%.0f s)' % (h1, h2, d, F.conformal_weights(), time.time() - t))

V(1/8) x V(-1/24), depth 1: [(0, [2]), (1, [3, 1]), (2, [2])]  (0 s)


V(-1/24) x V(-1/24), depth 2: [(0, [2]), (1/3, [2]), (1, [1]), (4/3, [2]), (2, [3, 1, 1]), (7/3, [2, 2]), (3, [2]), (4, [2, 2])]  (12 s)


## 6. Limitations

* **Fusion with staggered modules.** Most of Eberle and Flohr's rank-3 modules come from fusion products *with* rank-2 modules. `fusion_product` accepts any module, but the staggered modules $\mathcal R^{(2)}$ are not constructed in the library yet; they could be built from their generating states and null vectors, or as fusion products once these are modules.
* **Two-parameter couplings.** The couplings of $\mathcal R^{(2)}(1, 7)$ and $\mathcal R^{(2)}(2, 7)$ are not computed (section 4).
* **Windows.** The windows of section 1 are chosen from the levels of the generating null vectors found up to level 14. The code does not prove that a window is large enough, and stability in $K$ is only evidence.
* **Cost.** Large windows are expensive: the couplings at level 4 need $K = 12$ and 10–15 minutes each, and the table of section 3 takes about ten minutes.
* **Other augmented minimal models.** The augmented Yang–Lee model of central charge $c_{2,5} = -\frac{22}5$ [EF2006, Appendix C] is not treated here; the same code applies, with `va.Virasoro(QQ, -22/5, ...)` and the Kac table of $c_{2,5}$.

## References

- [EF2006] H. Eberle, M. Flohr, *Virasoro representations and fusion for general augmented minimal models*, J. Phys. A 39 (2006), 15245–15286, [arXiv:hep-th/0604097](https://arxiv.org/abs/hep-th/0604097). — the fusion products and logarithmic couplings reproduced here (sections 1–6).
- [Nah1994] W. Nahm, *Quasi-rational fusion products*, Internat. J. Modern Phys. B 8 (1994), 3693–3702, [arXiv:hep-th/9402039](https://arxiv.org/abs/hep-th/9402039). — the depth quotients and the special subspace (introduction, section 1).
- [GK1996] M. R. Gaberdiel, H. G. Kausch, *Indecomposable fusion products*, Nuclear Phys. B 477 (1996), 293–318, [arXiv:hep-th/9604026](https://arxiv.org/abs/hep-th/9604026). — the Nahm–Gaberdiel–Kausch algorithm (introduction).
- [GRW2009] M. R. Gaberdiel, I. Runkel, S. Wood, *Fusion rules and boundary conditions in the c = 0 triplet model*, J. Phys. A 42 (2009), 325403, [arXiv:0905.0916](https://arxiv.org/abs/0905.0916). — the correction of $\mathcal V(2) \boxtimes \mathcal V(2)$ and of the products of $\mathcal V(1)$ (section 2).
- [KyR2009] K. Kytölä, D. Ridout, *On staggered indecomposable Virasoro modules*, J. Math. Phys. 50 (2009), 123503, [arXiv:0905.0108](https://arxiv.org/abs/0905.0108). — staggered modules and their logarithmic couplings (section 4).